# Model Architecture Visualization

Pick a **model class** (registry key) and one of its **`conf/model/*.yaml`** configs with
the searchable dropdowns below, then render. Default backend is
[torchview](https://github.com/mert-kurttutan/torchview) (module-level graph with tensor
shapes); **torchviz**, **torchsummary**, **torchlens** and **visualtorch** remain available.

The model is built exactly as `train.py` builds it (`training.config.instantiate_model`) and
called through its task codec, so the dummy input is what the training loop would feed.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path
from typing import Any, Callable, Literal

import torch
import torch.nn as nn
from IPython.display import SVG, Image, display
from omegaconf import DictConfig, OmegaConf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for _p in (ROOT, ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from training.config import build_task_and_codec, instantiate_model

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------------
# conf/model/*.yaml -> {model class: [config paths]}
# The class key is whatever the `_target_` dispatches on: `params.name`
# (models.registry.build_model), `params.model_type` (build_legacy_inline),
# `params.model_name` (build_noise_gen_model) or the factory function itself.
# ---------------------------------------------------------------------------

_CLASS_KEYS = {
    "models.registry.build_model": "name",
    "models.registry.build_legacy_inline": "model_type",
    "models.registry.build_noise_gen_model": "model_name",
}


def model_class_of(cfg: DictConfig) -> str:
    target = str(cfg.get("_target_") or "")
    key = _CLASS_KEYS.get(target)
    if key is not None:
        return str(cfg.params[key])
    return target.rsplit(".", 1)[-1] or str(cfg.get("model_type"))


def scan_model_configs() -> dict[str, list[Path]]:
    classes: dict[str, list[Path]] = {}
    for path in sorted((ROOT / "conf" / "model").glob("*.yaml")):
        cfg = OmegaConf.load(path)
        classes.setdefault(model_class_of(cfg), []).append(path)
    return dict(sorted(classes.items()))


MODEL_CONFIGS = scan_model_configs()
print(f"{sum(map(len, MODEL_CONFIGS.values()))} configs across {len(MODEL_CONFIGS)} model classes")

In [ ]:
# ---------------------------------------------------------------------------
# Build model + dummy inputs through the task codec
# ---------------------------------------------------------------------------

NUM_ROTORS = 4


def _dummy_inputs(cfg: DictConfig, seconds: float) -> dict[str, torch.Tensor]:
    tp = cfg.get("task_params") or {}
    sr_num, sr_den = tp.get("sr", [16000, 1])
    sr = int(sr_num) // int(sr_den)
    # Legacy inline configs fix their STFT frame count via audio.chunk_size.
    chunk = (cfg.params.get("config") or {}).get("audio", {}).get("chunk_size")
    n = int(chunk) if chunk else int(sr * seconds)
    n_ch = tp.get("n_channels")
    mixture = torch.randn(1, n) if n_ch is None else torch.randn(1, int(n_ch), n)

    def frames(rate_key: str) -> int:
        num, den = tp.get(rate_key) or (sr, 1)
        return n * int(num) // (sr * int(den)) + 1

    task = cfg.task
    if task == "noise_generation":
        return {
            "rps": 40.0 + 10.0 * torch.rand(1, NUM_ROTORS, n),
            "rel_pos": torch.randn(1, NUM_ROTORS, 3),
            "drone_names": ["dregon"],
        }
    inputs = {"mixture": mixture}
    if task == "speech_enhancement" and tp.get("use_rps"):
        inputs["rps"] = 40.0 + 10.0 * torch.rand(1, NUM_ROTORS, frames("rps_rate"))
    if task == "rps_prediction" and tp.get("use_cond"):
        inputs["rps_cond"] = 40.0 + 10.0 * torch.rand(1, NUM_ROTORS, frames("frame_rate"))
    return inputs


class CodecCall(nn.Module):
    """`model` invoked the way the training loop invokes it (through the codec)."""

    def __init__(self, model: nn.Module, codec: Any):
        super().__init__()
        self.model = model
        self.codec = codec

    def forward(self, **inputs):
        return self.codec.call_model(self.model, inputs)


def _patch_flash_attn_for_cpu(model: nn.Module):
    try:
        from models.edge_bs_rof.attend import Attend
    except ImportError:
        return
    for m in model.modules():
        if isinstance(m, Attend):
            m.flash = False


def load(config_path: str | Path, device: str = "cpu", seconds: float = 1.0):
    cfg = OmegaConf.load(ROOT / config_path)
    model = instantiate_model(cfg).to(device).eval()
    if device == "cpu":
        _patch_flash_attn_for_cpu(model)
    _task, codec = build_task_and_codec(cfg)
    inputs = {
        k: (v.to(device) if isinstance(v, torch.Tensor) else v)
        for k, v in _dummy_inputs(cfg, seconds).items()
    }
    return cfg, CodecCall(model, codec), inputs

In [ ]:
# ---------------------------------------------------------------------------
# Backends
# ---------------------------------------------------------------------------

Backend = Literal["torchview", "torchviz", "torchsummary", "torchlens", "visualtorch"]
Renderer = Callable[[CodecCall, dict[str, Any], str, Path | None, int], None]


def _viz_torchview(wrapped, inputs, slug, save_path, depth):
    from torchview import draw_graph

    # `depth` counts from the model itself; the codec wrapper adds one level.
    # torchview trips (KeyError in render_edges) inside some third-party
    # submodules (nnAudio's CQT2010v2) once they are expanded; back off one
    # level at a time until the graph renders.
    for d in range(depth + 1, 1, -1):
        try:
            g = draw_graph(
                wrapped,
                input_data=inputs,
                depth=d,
                expand_nested=True,
                hide_inner_tensors=True,
                graph_dir="TB",
                graph_name=slug,
                device=next(wrapped.parameters()).device,
            )
            dot = g.visual_graph
            svg = dot.pipe(format="svg")
            break
        except KeyError:
            print(f"  torchview could not expand depth {d - 1}; trying {d - 2}")
    else:
        raise RuntimeError("torchview failed at every depth")
    if d - 1 != depth:
        print(f"  rendered at depth {d - 1}")
    if save_path:
        dot.render(str(save_path), format="png", cleanup=True)
    display(SVG(svg))


def _tensor_args(inputs: dict[str, Any]) -> list[torch.Tensor]:
    return [v for v in inputs.values() if isinstance(v, torch.Tensor)]


def _viz_torchviz(wrapped, inputs, slug, save_path, depth):
    from torchviz import make_dot

    out = wrapped(**inputs)
    while isinstance(out, (tuple, list, dict)):
        out = next(iter(out.values())) if isinstance(out, dict) else out[0]
    dot = make_dot(out, params=dict(wrapped.model.named_parameters()))
    dot.attr(rankdir="TB")
    if save_path:
        dot.render(str(save_path), format="png", cleanup=True)
    display(SVG(dot.pipe(format="svg")))


def _viz_torchsummary(wrapped, inputs, slug, save_path, depth):
    from torchsummary import summary

    (x,) = _tensor_args(inputs)[:1]
    summary(wrapped.model, input_size=tuple(x.shape[1:]), device=str(x.device))


def _viz_torchlens(wrapped, inputs, slug, save_path, depth):
    import torchlens as tl

    out_path = str(save_path) if save_path else f"/tmp/torchlens_{slug}"
    tl.log_forward_pass(
        wrapped.model,
        _tensor_args(inputs),
        layers_to_save="none",
        vis_opt="unrolled",
        vis_outpath=out_path,
        vis_fileformat="png",
        vis_save_only=True,
    )
    display(Image(filename=out_path + ".png"))


def _viz_visualtorch(wrapped, inputs, slug, save_path, depth):
    import visualtorch

    (x,) = _tensor_args(inputs)[:1]
    img = visualtorch.graph_view(wrapped.model, input_shape=list(x.shape[1:]))
    if save_path:
        img.save(str(save_path.with_suffix(".png")))
    display(img)


BACKENDS: dict[Backend, tuple[Renderer, str]] = {
    "torchview": (_viz_torchview, "Module graph with tensor shapes (default). `depth` controls nesting."),
    "torchviz": (_viz_torchviz, "Autograd data-flow DAG; graphviz layout can take minutes on conv stacks."),
    "torchsummary": (_viz_torchsummary, "Text table of layers / shapes / params. Single tensor in; breaks on LSTM tuple outputs."),
    "torchlens": (_viz_torchlens, "Full op graph with activation metadata."),
    "visualtorch": (_viz_visualtorch, "Layered diagram (PIL). Sequential-style models only."),
}


def visualize(
    config_path: str | Path,
    backend: Backend = "torchview",
    depth: int = 3,
    device: str = "cpu",
    seconds: float = 1.0,
    save_dir: str | Path | None = None,
):
    config_path = Path(config_path)
    slug = config_path.stem
    cfg, wrapped, inputs = load(config_path, device, seconds)
    n_params = sum(p.numel() for p in wrapped.model.parameters())
    shapes = {k: tuple(v.shape) for k, v in inputs.items() if isinstance(v, torch.Tensor)}
    print(f"{model_class_of(cfg)}  [{config_path.name}]  task={cfg.task}")
    print(f"backend={backend}  params={n_params:,}  inputs={shapes}")
    save_path = None
    if save_dir:
        save_path = Path(save_dir) / f"{slug}_{backend}"
        save_path.parent.mkdir(parents=True, exist_ok=True)
    BACKENDS[backend][0](wrapped, inputs, slug, save_path, depth)

In [ ]:
# ---------------------------------------------------------------------------
# UI: searchable class / config dropdowns
# ---------------------------------------------------------------------------
import ipywidgets as W

_classes = list(MODEL_CONFIGS)
_class = W.Combobox(
    options=_classes, value="hppnet_orig" if "hppnet_orig" in _classes else _classes[0],
    description="class", ensure_option=True, placeholder="type to search", layout=W.Layout(width="420px"),
)
_config = W.Combobox(description="config", ensure_option=True, placeholder="type to search", layout=W.Layout(width="420px"))
_backend = W.Dropdown(options=list(BACKENDS), value="torchview", description="backend")
_depth = W.IntSlider(value=3, min=1, max=8, description="depth")
_device = W.Dropdown(options=["cpu"] + (["cuda"] if torch.cuda.is_available() else []), value="cpu", description="device")
_run = W.Button(description="Render", button_style="primary")
_out = W.Output()


def _sync_configs(*_):
    paths = MODEL_CONFIGS.get(_class.value, [])
    _config.options = [str(p.relative_to(ROOT)) for p in paths]
    _config.value = _config.options[0] if _config.options else ""


def _on_run(_):
    with _out:
        _out.clear_output(wait=True)
        if not _config.value:
            print("pick a config")
            return
        visualize(_config.value, backend=_backend.value, depth=_depth.value, device=_device.value)


_class.observe(_sync_configs, names="value")
_run.on_click(_on_run)
_sync_configs()

display(W.VBox([W.HBox([_class, _config]), W.HBox([_backend, _depth, _device, _run]), _out]))

## Programmatic use

```python
visualize("conf/model/hppnet_l2.yaml")                       # torchview, depth 3
visualize("conf/model/hb_scv2_mag.yaml", depth=5)
visualize("conf/model/a1_baseline_dcunet.yaml", backend="torchsummary")
visualize("conf/model/hppnet_rps_l4.yaml", save_dir="outputs/viz")
MODEL_CONFIGS["simple_conv_v2_transformer"]                  # configs of one class
```

In [ ]:
visualize("conf/model/harmof0_rps_l4.yaml", depth=3)

In [ ]:
visualize("conf/model/harmof0_l2.yaml", depth=2)

In [ ]:
visualize("conf/model/hppnet_rps.yaml")

In [ ]:
visualize("conf/model/hppnet_rps_l4.yaml")